# NB09 — The Monkey-Patching System

**North star callback:** When you call `FastLanguageModel.from_pretrained`, Unsloth
replaces HuggingFace model methods with its own Triton-fused equivalents — before the
model ever runs a forward pass. This is monkey-patching: Python attribute assignment
at the class level. No subclassing, no wrappers — just `Class.method = faster_fn`.

## 1. Python monkey-patching mechanics

In [1]:
import torch
import torch.nn as nn

# Save the original before touching anything — this is the pattern Unsloth uses too.
ORIGINAL_LINEAR_FORWARD = nn.Linear.forward

# --- Define a replacement ---
call_log = []

def logging_forward(self, x: torch.Tensor) -> torch.Tensor:
    call_log.append(f'Linear({self.in_features}→{self.out_features})')
    return ORIGINAL_LINEAR_FORWARD(self, x)

# --- Apply: one line, affects every nn.Linear instance everywhere ---
nn.Linear.forward = logging_forward

net = nn.Sequential(
    nn.Linear(256, 128, device='cuda', dtype=torch.bfloat16),
    nn.Linear(128,  64, device='cuda', dtype=torch.bfloat16),
)
x = torch.randn(4, 256, device='cuda', dtype=torch.bfloat16)
_ = net(x)

print(f'Calls intercepted : {call_log}')
print(f'nn.Linear.forward is original: {nn.Linear.forward is ORIGINAL_LINEAR_FORWARD}')

# --- Revert: equally simple ---
nn.Linear.forward = ORIGINAL_LINEAR_FORWARD
call_log.clear()
print(f'After revert — is original: {nn.Linear.forward is ORIGINAL_LINEAR_FORWARD}')

Calls intercepted : ['Linear(256→128)', 'Linear(128→64)']
nn.Linear.forward is original: False
After revert — is original: True


## 2. A minimal patch registry

Unsloth groups its patches under model-specific `pre_patch()` static methods.
Here we build the same idea as a reusable registry with apply / revert semantics.

In [2]:
class PatchRegistry:
    """Register (cls, method_name, replacement) triples; apply and revert atomically."""

    def __init__(self):
        self._patches   = []   # list of (cls, method_name, replacement)
        self._originals = {}   # (cls, method_name) -> original

    def register(self, cls, method_name: str, replacement):
        self._patches.append((cls, method_name, replacement))

    def apply(self):
        for cls, method, replacement in self._patches:
            self._originals[(cls, method)] = getattr(cls, method)
            setattr(cls, method, replacement)
            print(f'  patched {cls.__name__}.{method}')

    def revert(self):
        for (cls, method), original in self._originals.items():
            setattr(cls, method, original)
        self._originals.clear()
        print(f'  reverted {len(self._patches)} patch(es)')


# --- Demo: dtype-enforcing patch ---
def bf16_only_forward(self, x: torch.Tensor) -> torch.Tensor:
    if x.dtype != torch.bfloat16:
        raise TypeError(f'Expected bfloat16 input, got {x.dtype}')
    return ORIGINAL_LINEAR_FORWARD(self, x)


registry = PatchRegistry()
registry.register(nn.Linear, 'forward', bf16_only_forward)

print('Applying patches:')
registry.apply()

linear = nn.Linear(64, 32, device='cuda', dtype=torch.bfloat16)

# bf16 passes
_ = linear(torch.randn(2, 64, device='cuda', dtype=torch.bfloat16))
print('  bf16 input: OK')

# fp32 is rejected
try:
    _ = linear(torch.randn(2, 64, device='cuda', dtype=torch.float32))
except TypeError as e:
    print(f'  fp32 input: caught — {e}')

print('Reverting:')
registry.revert()

# Confirm the revert worked — cast weights to fp32 so fp32 input matches
linear = linear.float()
_ = linear(torch.randn(2, 64, device='cuda', dtype=torch.float32))
print('  fp32 after revert: OK')

Applying patches:
  patched Linear.forward
  bf16 input: OK
  fp32 input: caught — Expected bfloat16 input, got torch.float32
Reverting:
  reverted 1 patch(es)
  fp32 after revert: OK


## 3. Unsloth's patching structure

Unsloth ships a model file per architecture (`llama.py`, `mistral.py`, …).
Each defines a `FastXxxModel` class with two static methods:
- `pre_patch()` — swaps HF class methods before model weights are loaded
- `post_patch(model, tokenizer)` — applies weight-level fixes after loading

The entry point is `FastLanguageModel.from_pretrained`, which dispatches
to the right `FastXxxModel` based on the model's config.

In [3]:
import os

models_dir = '../unsloth/unsloth/models'
print('unsloth/models/ — per-architecture patch files:')
for fname in sorted(os.listdir(models_dir)):
    if fname.endswith('.py') and not fname.startswith('_'):
        path = os.path.join(models_dir, fname)
        lines = sum(1 for _ in open(path))
        print(f'  {fname:30s} {lines:5d} lines')

unsloth/models/ — per-architecture patch files:
  cohere.py                        538 lines
  dpo.py                            26 lines
  falcon_h1.py                     773 lines
  gemma.py                         493 lines
  gemma2.py                        662 lines
  glm4_moe.py                      450 lines
  granite.py                       634 lines
  llama.py                        3614 lines
  llama4.py                         16 lines
  loader.py                       1643 lines
  loader_utils.py                  433 lines
  mapper.py                       1466 lines
  mistral.py                       487 lines
  qwen2.py                         101 lines
  qwen3.py                         478 lines
  qwen3_moe.py                     243 lines
  rl.py                           2076 lines
  rl_replacements.py              1651 lines
  sentence_transformer.py         2111 lines
  vision.py                       1662 lines


## 4. Read Unsloth's pre_patch for LLaMA

The LLaMA `pre_patch()` is the clearest example of the pattern:
it replaces six HuggingFace class methods in 7 lines.

In [4]:
import sys, inspect
sys.path.insert(0, '../unsloth')

from unsloth.models.llama import FastLlamaModel
print(inspect.getsource(FastLlamaModel.pre_patch))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
    @staticmethod
    def pre_patch():
        init_name, function = patch_llama_rope_scaling(
            model_name = "llama",
            rope_module = LlamaRotaryEmbedding,
            scaled_rope_module = LlamaLinearScalingRotaryEmbedding,
            extended_rope_module = LlamaExtendedRotaryEmbedding,
            attention_module = LlamaAttention,
            longrope_module = LongRopeRotaryEmbedding,
        )
        if init_name is not None:
            exec(function, globals())
            LlamaAttention.__init__ = eval(init_name)
        LlamaAttention.forward = LlamaAttention_fast_forward
        LlamaSdpaAttention.forward = LlamaAttention_fast_forward
        LlamaFlashAttention2.forward = LlamaAttention_fast_for

## 5. patch_fast_lora — patching a PEFT class

`patch_fast_lora()` in `_utils.py` replaces PEFT's `Linear4bit.forward`
with Unsloth's fused version — the same function we inspected in NB07.
It's three lines: import, one setattr, done.

In [5]:
from unsloth.models._utils import patch_fast_lora
print(inspect.getsource(patch_fast_lora))

def patch_fast_lora():
    import peft.tuners.lora.bnb

    peft.tuners.lora.bnb.Linear4bit.forward = fast_lora_forward



## 6. Write a custom patch — latency tracer

To add your own optimization: define the replacement, register it,
then call `apply()` before the model runs. Unsloth follows this exact pattern.

In [6]:
import sys
sys.path.insert(0, '..')
from utils.benchmark import compare

import time

# --- Custom patch: latency tracer for nn.Linear ---
trace_log = []

def latency_tracing_forward(self, x: torch.Tensor) -> torch.Tensor:
    t0 = time.perf_counter()
    out = ORIGINAL_LINEAR_FORWARD(self, x)
    torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) * 1000
    trace_log.append((f'{self.in_features}→{self.out_features}', ms))
    return out


# A small model that resembles one transformer MLP block
class MiniMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.gate = nn.Linear(4096, 14336, bias=False, device='cuda', dtype=torch.bfloat16)
        self.up   = nn.Linear(4096, 14336, bias=False, device='cuda', dtype=torch.bfloat16)
        self.down = nn.Linear(14336, 4096, bias=False, device='cuda', dtype=torch.bfloat16)

    def forward(self, x):
        gate = torch.nn.functional.silu(self.gate(x))
        return self.down(gate * self.up(x))


mlp = MiniMLP()
x   = torch.randn(2, 512, 4096, device='cuda', dtype=torch.bfloat16)

# Apply patch
nn.Linear.forward = latency_tracing_forward

# Warmup + trace
for _ in range(3):
    trace_log.clear()
    _ = mlp(x)

# Revert
nn.Linear.forward = ORIGINAL_LINEAR_FORWARD

print('MiniMLP forward — per-layer latency (last run):')
for name, ms in trace_log:
    print(f'  Linear {name}: {ms:.3f} ms')
print(f'  Total : {sum(ms for _, ms in trace_log):.3f} ms')

MiniMLP forward — per-layer latency (last run):
  Linear 4096→14336: 0.791 ms
  Linear 4096→14336: 0.947 ms
  Linear 14336→4096: 0.968 ms
  Total : 2.706 ms


## 7. Exercises

1. **Patch count by layer type**: extend the `PatchRegistry` to also patch
   `nn.LayerNorm.forward` and `nn.Embedding.forward` with counting wrappers.
   Run a `MiniMLP` forward pass and print a call-count table per layer type.

2. **Pre-patch hook**: modify `PatchRegistry.apply` to accept an optional
   `before_fn(cls, method_name)` callback. Use it to log a warning if a method
   has already been patched by another registry (i.e., the current attribute
   is not the function's original `__wrapped__` or module default).

3. **Write a real optimization patch**: using the fused RoPE kernel from NB05,
   replace `transformers.models.llama.modeling_llama.apply_rotary_pos_emb`
   with `apply_rope_fused`. Verify correctness with a small input and measure
   the latency improvement (you will need to adapt the signature to match
   HuggingFace's `(q, k, cos, sin, position_ids, unsqueeze_dim)` API).